# 9. Feature Selection

## Purpose
Remove low-information features from each normalized profile using pycytominer's
`feature_select`. Feature selection is **fit on a reference subset** (DMSO and
Staurosporine wells) but **applied to all treatments**, so the retained feature set
is determined by the reference population and then used to subset the full dataset.

We fit on DMSO and Staurosporine wells because we expect that these two treatments will have the most distinct profiles, so features that are uninformative in this context are likely to be uninformative across the full treatment set. 
By fitting on this reference subset, we can identify and retain features that capture meaningful variation while removing those that do not contribute to distinguishing between treatments.

This is **step 9 of Stage 4 (image-based profiling)**. It runs once per patient
and must follow `8.normalization.ipynb`.

## Inputs

Six normalized parquets from `5.normalized_profiles/`:

| File | Profile type |
|---|---|
| `sc_norm.parquet` | Hand-crafted SC |
| `organoid_norm.parquet` | Hand-crafted organoid |
| `sammed_sc_norm.parquet` | Deep-learning SC (SAMMed3D) |
| `sammed_organoid_norm.parquet` | Deep-learning organoid (SAMMed3D) |
| `sammed_nucleocentric_norm.parquet` | Deep-learning nucleocentric (SAMMed3D) |
| `nucleocentric_morphem_norm.parquet` | Deep-learning nucleocentric (morphem) |

## Outputs

Six feature-selected parquets in `6.feature_selected_profiles/`:

| File | Profile type |
|---|---|
| `sc_fs.parquet` | Hand-crafted SC |
| `organoid_fs.parquet` | Hand-crafted organoid |
| `sammed_sc_fs.parquet` | Deep-learning SC (SAMMed3D) |
| `sammed_organoid_fs.parquet` | Deep-learning organoid (SAMMed3D) |
| `sammed_nucleocentric_fs.parquet` | Deep-learning nucleocentric (SAMMed3D) |
| `nucleocentric_morphem_fs.parquet` | Deep-learning nucleocentric (morphem) |

## Notes
- Feature selection is fit on DMSO and Staurosporine rows only, then the retained
  feature set is applied back to the full (all-treatment) dataset. This ensures
  feature selection is not biased by the full treatment distribution.
- QC-flagged rows are not filtered here; that is left to downstream analysis.
- `drop_outliers` (`outlier_cutoff=100`) was added per
  [#163](https://github.com/WayScience/NF1_3D_organoid_profiling_pipeline/issues/163).
- `variance_threshold` and `frequency_threshold` are two distinct pycytominer
  operations (pycytominer>=1.7): `variance_threshold` only uses `min_variance`
  (left at pycytominer's own default), while `freq_cut`/`unique_cut` are consumed
  by `frequency_threshold` specifically. The previous operation list included only
  `"variance_threshold"`, so `freq_cut`/`unique_cut` were silently unused --
  confirmed directly (a synthetic low-frequency feature survived feature selection
  under the old config). `"frequency_threshold"` is now included explicitly.

In [1]:
import os
import pathlib

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from pycytominer import feature_select

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
# NOTE: previously this line unconditionally overrode bandicoot_check()
# with root_dir, meaning bandicoot was never actually used even when
# mounted. Removed so bandicoot_check()'s own bandicoot-first behavior
# takes effect.

In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0037_T1_CQ1"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
## Pathing
sc_normalized_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/sc_norm.parquet"
).resolve(strict=True)
organoid_normalized_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/organoid_norm.parquet"
).resolve(strict=True)
# The 4 deep-learning inputs are optional: a dataset with no deep-learning
# features (e.g. ZEDProfiler-only) never has 8.normalization.py produce these
# files, so each is only resolved here if it actually exists.
sc_sammed_normalized_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/sammed_sc_norm.parquet"
).resolve()
organoid_sc_sammed_normalized_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/sammed_organoid_norm.parquet"
).resolve()
nucleocentric_sammed_normalized_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/sammed_nucleocentric_norm.parquet"
).resolve()
nucleocentric_morphem_normalized_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/5.normalized_profiles/nucleocentric_morphem_norm.parquet"
).resolve()
has_sc_sammed = sc_sammed_normalized_path.exists()
has_organoid_sammed = organoid_sc_sammed_normalized_path.exists()
has_nucleocentric_sammed = nucleocentric_sammed_normalized_path.exists()
has_nucleocentric_morphem = nucleocentric_morphem_normalized_path.exists()


# output path
sc_fs_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/sc_fs.parquet"
).resolve()
organoid_fs_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/organoid_fs.parquet"
).resolve()
sc_sammed_feature_selected_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/sammed_sc_fs.parquet"
).resolve()
organoid_sc_sammed_feature_selected_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/sammed_organoid_fs.parquet"
).resolve()
nucleocentric_sammed_feature_selected_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/sammed_nucleocentric_fs.parquet"
).resolve()
nucleocentric_morphem_feature_selected_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/6.feature_selected_profiles/nucleocentric_morphem_fs.parquet"
).resolve()

organoid_fs_output_path.parent.mkdir(parents=True, exist_ok=True)

In [4]:
# read in the data
sc_normalized = pd.read_parquet(sc_normalized_path)
organoid_normalized = pd.read_parquet(organoid_normalized_path)
print(f"SC normalized loaded. Shape: {sc_normalized.shape}")
print(f"Organoid normalized loaded. Shape: {organoid_normalized.shape}")

run_dict = {
    "sc_normalized": {
        "df": sc_normalized,
        "output_path": sc_fs_output_path,
    },
    "organoid_normalized": {
        "df": organoid_normalized,
        "output_path": organoid_fs_output_path,
    },
}

# The 4 deep-learning profile types are only added to run_dict (and therefore
# feature-selected below) when this dataset actually produced them -- absent
# for datasets with no deep-learning features (e.g. ZEDProfiler-only).
if has_sc_sammed:
    sc_sammed_normalized = pd.read_parquet(sc_sammed_normalized_path)
    print(f"SAMMed3D SC normalized loaded. Shape: {sc_sammed_normalized.shape}")
    run_dict["sc_sammed"] = {
        "df": sc_sammed_normalized,
        "output_path": sc_sammed_feature_selected_output_path,
    }
if has_organoid_sammed:
    organoid_sc_sammed_normalized = pd.read_parquet(organoid_sc_sammed_normalized_path)
    print(
        f"SAMMed3D organoid normalized loaded. Shape: {organoid_sc_sammed_normalized.shape}"
    )
    run_dict["organoid_sc_sammed"] = {
        "df": organoid_sc_sammed_normalized,
        "output_path": organoid_sc_sammed_feature_selected_output_path,
    }
if has_nucleocentric_sammed:
    nucleocentric_sammed_normalized = pd.read_parquet(
        nucleocentric_sammed_normalized_path
    )
    print(
        f"SAMMed3D nucleocentric normalized loaded. Shape: {nucleocentric_sammed_normalized.shape}"
    )
    run_dict["nucleocentric_sammed"] = {
        "df": nucleocentric_sammed_normalized,
        "output_path": nucleocentric_sammed_feature_selected_output_path,
    }
if has_nucleocentric_morphem:
    nucleocentric_morphem_normalized = pd.read_parquet(
        nucleocentric_morphem_normalized_path
    )
    print(
        f"morphem nucleocentric normalized loaded. Shape: {nucleocentric_morphem_normalized.shape}"
    )
    run_dict["nucleocentric_chammi"] = {
        "df": nucleocentric_morphem_normalized,
        "output_path": nucleocentric_morphem_feature_selected_output_path,
    }

SC normalized loaded. Shape: (15058, 2668)
Organoid normalized loaded. Shape: (1336, 912)
SAMMed3D SC normalized loaded. Shape: (14827, 9242)
SAMMed3D organoid normalized loaded. Shape: (1837, 3095)
SAMMed3D nucleocentric normalized loaded. Shape: (14917, 1562)
morphem nucleocentric normalized loaded. Shape: (14917, 1562)


In [5]:
# Feature selection operations applied in order:
#   drop_na_columns       — remove features with >na_cutoff fraction of NaN values
#   drop_outliers         — remove features whose min or max absolute value exceeds
#                         outlier_cutoff (see WayScience/NF1_3D_organoid_profiling_pipeline#163)
#   blocklist             — remove features on the pycytominer blocklist (known noisy/artifactual)
#   variance_threshold    — remove near-constant features (variance below pycytominer's
#                         own min_variance default; not overridden here)
#   frequency_threshold   — remove features with a large most-common/second-most-common
#                         value gap (freq_cut) or too few unique values relative to
#                         sample count (unique_cut, left at pycytominer's own default)
#   correlation_threshold — remove one feature from each pair with Pearson r > corr_threshold
feature_select_ops = [
    "drop_na_columns",
    "drop_outliers",
    "blocklist",
    "variance_threshold",
    "frequency_threshold",
    "correlation_threshold",  # comment out to remove correlation thresholding
]
na_cutoff = 0.05  # drop features with >5% NaN
outlier_cutoff = 100  # drop features whose min/max absolute value exceeds this
corr_threshold = 0.90  # drop one of any pair with Pearson r >= 0.90
freq_cut = 0.05  # frequency threshold: most-common / second-most-common value ratio
# unique_cut (frequency_threshold) and min_variance (variance_threshold) are
# intentionally left at pycytominer's own defaults (0.01 and 1e-6) -- not overridden.

## Feature select the profiles

For each profile type:
1. Feature selection is **fit** on DMSO and Staurosporine rows only — a controlled
   reference that avoids biasing feature selection on the full treatment distribution.
2. The retained feature set is applied back to the **full dataset** (all treatments),
   so no treatment rows are dropped from the output.

In [6]:
for profile_name in run_dict.keys():
    print(f"Running feature selection for {profile_name} profiles...")
    df = run_dict[profile_name]["df"]
    output_path = run_dict[profile_name]["output_path"]
    # prep profiles for feature selection
    # grab feature columns
    features_columns = [col for col in df.columns if not col.startswith("Metadata_")]
    df[features_columns] = df[features_columns].replace([np.inf, -np.inf], np.nan)
    # Phase 1: fit feature selection on reference treatments only.
    # select DMSO 1% and Staurosporine 10 nM conditions for feature selection
    # run feature selection
    fs_profiles = feature_select(
        df,
        operation=feature_select_ops,
        features=features_columns,
        na_cutoff=na_cutoff,
        outlier_cutoff=outlier_cutoff,
        corr_threshold=corr_threshold,
        freq_cut=freq_cut,
        samples="(Metadata_Experiment_Treatment == 'DMSO' and Metadata_Experiment_Dose == 1) or (Metadata_Experiment_Treatment == 'Staurosporine' and Metadata_Experiment_Dose == 10)",
        output_file=output_path,
        output_type="parquet",
    )

    original_data_shape = df.shape
    parquet_file = pq.ParquetFile(fs_profiles)
    num_rows = parquet_file.metadata.num_rows
    num_columns = len(parquet_file.schema.names)
    fs_shape = (num_rows, num_columns)
    print("The number features before feature selection:", original_data_shape[1])
    print("The number features after feature selection:", fs_shape[1])

Running feature selection for sc_normalized profiles...
The number features before feature selection: 2668
The number features after feature selection: 407
Running feature selection for organoid_normalized profiles...
The number features before feature selection: 912
The number features after feature selection: 177
Running feature selection for sc_sammed profiles...
The number features before feature selection: 9242
The number features after feature selection: 5913
Running feature selection for organoid_sc_sammed profiles...
The number features before feature selection: 3095
The number features after feature selection: 1950
Running feature selection for nucleocentric_sammed profiles...
The number features before feature selection: 1562
The number features after feature selection: 1561
Running feature selection for nucleocentric_chammi profiles...
The number features before feature selection: 1562
The number features after feature selection: 1562
